In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, BertForTokenClassification, AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


# BERT-based NER class
class BertNER(nn.Module):
    def __init__(self, tokens_dim):
        super(BertNER, self).__init__()
        
        if type(tokens_dim) != int:
            raise TypeError('tokens_dim should be an integer')
        if tokens_dim <= 0:
            raise ValueError('Classification layer dimension should be at least 1')

        self.pretrained = BertForTokenClassification.from_pretrained("bert-base-cased", num_labels=tokens_dim)

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Forward pass of the network.
        """
        if labels is None:
            return self.pretrained(input_ids=input_ids, attention_mask=attention_mask)
        return self.pretrained(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    
    def save_pretrained(self, save_directory):
        """
        Save the pretrained model to the specified directory.
        """
        self.pretrained.save_pretrained(save_directory)

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

# roberta

In [10]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, RobertaForTokenClassification, AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


# RoBERTa-based NER class
class RobertaNER(nn.Module):
    def __init__(self, tokens_dim):
        super(RobertaNER, self).__init__()
        
        if type(tokens_dim) != int:
            raise TypeError('tokens_dim should be an integer')
        if tokens_dim <= 0:
            raise ValueError('Classification layer dimension should be at least 1')

        # Use RoBERTa large model for token classification
        self.pretrained = RobertaForTokenClassification.from_pretrained("roberta-large", num_labels=tokens_dim)

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Forward pass of the network.
        """
        if labels is None:
            return self.pretrained(input_ids=input_ids, attention_mask=attention_mask)
        return self.pretrained(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
    
    def save_pretrained(self, save_directory):
        """
        Save the pretrained model to the specified directory.
        """
        self.pretrained.save_pretrained(save_directory)


# Initialize the tokenizer for RoBERTa
tokenizer = AutoTokenizer.from_pretrained("roberta-large", add_prefix_space=True)

In [2]:
import torch
from torch.utils.data import Dataset
import ast

class NerDataset(Dataset):
    """
    Custom dataset class for NER tasks.
    """

    def __init__(self, dataframe, tokenizer, label_map, max_len=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        row = self.data.iloc[index]

        # Ensure the 'Ingredient_Phrases' column is a string
        text = str(row['Ingredient_Phrases'])

        # Convert 'Labels' from string representation to a list
        labels_list = ast.literal_eval(row['Labels'])  

        encoding = self.tokenizer(
            text.split(),  # Tokenize word by word
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
            is_split_into_words=True  # Ensure token alignment with words
        )

        word_ids = encoding.word_ids(batch_index=0)  # Get token-word alignment
        label_ids = [-100] * self.max_len  # Default to ignored token

        # Assigning NER labels for each word token
        previous_word_id = None
        for i, word_id in enumerate(word_ids):
            if word_id is None:
                label_ids[i] = -100  # Padding or special tokens
            elif word_id != previous_word_id:
                label_ids[i] = self.label_map.get(labels_list[word_id], -100)  # Assign the correct label
            else:
                label_ids[i] = label_ids[i-1]  # For subwords, keep same label
            previous_word_id = word_id

        input_ids = encoding['input_ids'].squeeze(0)  # Remove batch dimension
        attention_mask = encoding['attention_mask'].squeeze(0)  # Remove batch dimension
        label_ids = torch.tensor(label_ids, dtype=torch.long)  # Convert to tensor

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': label_ids
        }

# Label mapping 
label_map = {
    'QUANTITY': 0,
    'UNIT': 1,
    'NAME': 2,
    'FORM': 3,
    'STATE': 4,
    'DF': 5,
    'SIZE': 6,
    # 'PREPROCESSING': 7,
    'O': -100  # Outside token or padding
}

In [11]:
from sklearn.model_selection import train_test_split
df = pd.read_csv('/home/ritisha21089/btp/data/10k_data_modified.csv')

#ner_dataset = NerDataset(df, tokenizer, label_map)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
test_dataset = NerDataset(dataframe=test_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


model = BertNER(tokens_dim=len(label_map))
# model = RobertaNER(tokens_dim=len(label_map))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
from sklearn.model_selection import train_test_split
df = pd.read_csv('/home/ritisha21089/btp/data/10k_data.csv')

#ner_dataset = NerDataset(df, tokenizer, label_map)
train_df_, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df, val_df = train_test_split(train_df_, test_size=0.2, random_state=42)

train_dataset = NerDataset(dataframe=train_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
test_dataset = NerDataset(dataframe=test_df, tokenizer=tokenizer, label_map=label_map, max_len=128)
val_dataset = NerDataset(dataframe=val_df, tokenizer=tokenizer, label_map=label_map, max_len=128)

batch_size = 16

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)


model = BertNER(tokens_dim=len(label_map))
# model = RobertaNER(tokens_dim=len(label_map))
# model = DebertaNER(tokens_dim=len(label_map))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training settings
epochs = 5
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()

/home/ritisha21089/miniconda3/lib/python3.9/site-packages/transformers/optimization.py:640: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


BertNER(
  (pretrained): BertForTokenClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(28996, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_features=768, 

- training with early stopping

In [8]:
import torch
import numpy as np
from tqdm import tqdm

# Train
train_losses = []
val_losses = []
patience = 3  # Stop training if validation loss does not improve for 'patience' epochs
best_val_loss = np.inf  # Initialize best validation loss as infinity
early_stop_counter = 0  # Tracks epochs without improvement
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

    # Training Loop
    model.train()
    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update tqdm description with latest loss
        progress_bar.set_postfix(loss=loss.item())

    avg_train_loss = total_loss / len(train_loader)
    train_losses.append(avg_train_loss)  # Store training loss

    # Validation Step
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_loader:  # Assuming you have a validation DataLoader
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_val_loss += loss.item()

    avg_val_loss = total_val_loss / len(val_loader)
    val_losses.append(avg_val_loss)  # Store validation loss

    print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")

    # Early Stopping Check
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss  # Update best validation loss
        early_stop_counter = 0  # Reset counter
        torch.save(model.state_dict(), "best_model.pth")  # Save best model
    else:
        early_stop_counter += 1
        print(f"Early stopping counter: {early_stop_counter}/{patience}")

    if early_stop_counter >= patience:
        print("Early stopping triggered! Stopping training.")
        break  # Stop training

Epoch 1/10, Train Loss: 0.1819, Validation Loss: 0.0735


Epoch 2/10, Train Loss: 0.0551, Validation Loss: 0.0731


Epoch 3/10, Train Loss: 0.0381, Validation Loss: 0.0743
Early stopping counter: 1/3


Epoch 4/10, Train Loss: 0.0289, Validation Loss: 0.0749
Early stopping counter: 2/3


Epoch 5/10, Train Loss: 0.0220, Validation Loss: 0.0818
Early stopping counter: 3/3
Early stopping triggered! Stopping training.


In [14]:
from tqdm import tqdm

# Train
train_losses = []

for epoch in range(epochs):
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update tqdm description with the latest loss
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)  # Store loss

    print(f"Epoch {epoch+1}/{epochs}, Average Loss: {avg_loss:.4f}")

Epoch 1/5, Average Loss: 0.0715


Epoch 2/5, Average Loss: 0.0444


Epoch 3/5, Average Loss: 0.0328


Epoch 4/5, Average Loss: 0.0255


Epoch 5/5, Average Loss: 0.0199


In [ ]:
# train
train_losses = []

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)  # Store loss
    print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")

In [15]:
# Evaluation 
model.eval()
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        logits = logits.detach().cpu().numpy()
        label_ids = labels.cpu().numpy()

        for i in range(len(label_ids)):
            pred_labels = []
            true_label = []
            for j in range(len(label_ids[i])):
                if label_ids[i][j] != -100:
                    true_label.append(label_ids[i][j])
                    pred_label = np.argmax(logits[i][j])
                    pred_labels.append(pred_label)
            predictions.append(pred_labels)
            true_labels.append(true_label)

- new results

In [16]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9733
Recall: 0.9717
F1-Score: 0.9724
              precision    recall  f1-score   support

           0     0.9953    0.9970    0.9962      3637
           1     0.9950    0.9947    0.9949      2626
           2     0.9907    0.9786    0.9846      6212
           3     0.9199    0.9189    0.9194       987
           4     0.9642    0.9903    0.9771      3205
           5     0.9859    0.9859    0.9859       213
           6     0.9617    0.9362    0.9488       564

    accuracy                         0.9823     17444
   macro avg     0.9733    0.9717    0.9724     17444
weighted avg     0.9824    0.9823    0.9823     17444



- new results 2.0

In [7]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9602
Recall: 0.9668
F1-Score: 0.9634
              precision    recall  f1-score   support

           0     0.9960    0.9945    0.9952      3476
           1     0.9928    0.9917    0.9922      2515
           2     0.9892    0.9794    0.9843      5725
           3     0.8966    0.9376    0.9167       962
           4     0.9027    0.9351    0.9186      1310
           5     0.9845    1.0000    0.9922       190
           6     0.9618    0.9522    0.9570       502
           7     0.9579    0.9439    0.9509      1640

    accuracy                         0.9743     16320
   macro avg     0.9602    0.9668    0.9634     16320
weighted avg     0.9748    0.9743    0.9745     16320



In [17]:
import torch

# Define checkpoint path
checkpoint_path = "bert_ner_new.pth"

# Save the model and optimizer state
torch.save({
    'epoch': epochs,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, checkpoint_path)

print(f"Checkpoint saved at {checkpoint_path}")

Checkpoint saved at bert_ner_new.pth


In [ ]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# Define checkpoint path
checkpoint_path = "bert_ner.pth"

model_new = BertNER(tokens_dim=len(label_map))

# Load the checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)

# Load model and optimizer state
model_new.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Ensure model_new is in evaluation mode
model_new.to(device)
model_new.eval()

print("Model loaded successfully and set to evaluation mode!")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_1456998/2733505170.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We rec

Model loaded successfully and set to evaluation mode!


In [15]:
predictions = []
true_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model_new(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        logits = logits.detach().cpu().numpy()
        label_ids = labels.cpu().numpy()

        for i in range(len(label_ids)):
            pred_labels = []
            true_label = []
            for j in range(len(label_ids[i])):
                if label_ids[i][j] != -100:  # Ignore padding labels
                    true_label.append(label_ids[i][j])
                    pred_label = np.argmax(logits[i][j])
                    pred_labels.append(pred_label)
            predictions.append(pred_labels)
            true_labels.append(true_label)

# Flatten lists for metric calculation
flat_predictions_new = [item for sublist in predictions for item in sublist]
flat_true_labels_new = [item for sublist in true_labels for item in sublist]

# Compute metrics
precision = precision_score(flat_true_labels_new, flat_predictions_new, average='macro')
recall = recall_score(flat_true_labels_new, flat_predictions_new, average='macro')
f1 = f1_score(flat_true_labels_new, flat_predictions_new, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

# Generate classification report
report = classification_report(flat_true_labels_new, flat_predictions_new, digits=4)
print(report)

Precision: 0.9574
Recall: 0.9648
F1-Score: 0.9610
              precision    recall  f1-score   support

           0     0.9960    0.9925    0.9942      3476
           1     0.9924    0.9920    0.9922      2515
           2     0.9884    0.9836    0.9860      5725
           3     0.9083    0.9366    0.9222       962
           4     0.9077    0.9160    0.9119      1310
           5     0.9744    1.0000    0.9870       190
           6     0.9430    0.9562    0.9496       502
           7     0.9490    0.9415    0.9452      1640

    accuracy                         0.9737     16320
   macro avg     0.9574    0.9648    0.9610     16320
weighted avg     0.9739    0.9737    0.9738     16320



In [18]:
import pandas as pd

df_results = pd.DataFrame({
    "True_Label": flat_true_labels,
    "Predicted_Label": flat_predictions
})

df_results.to_csv("bert_ner_predictions_new.csv", index=False)
print("Predictions saved to bert_ner_predictions.csv!")

Predictions saved to bert_ner_predictions.csv!


In [19]:
import torch

# Define test dataset save path
test_dataset_path = "test_loader_new.pth"

# Save the dataset (which is used in test_loader)
torch.save(test_dataset, test_dataset_path)

print(f"Test dataset saved at {test_dataset_path}")

Test dataset saved at test_loader_new.pth


In [14]:
# Load the test dataset
test_dataset = torch.load(test_dataset_path)

test_loader = DataLoader(test_dataset, batch_size=batch_size)

print("Test loader reloaded successfully!")

Test loader reloaded successfully!


/tmp/ipykernel_1456998/613144014.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  test_dataset = torch.load(test_dataset_path)


- 7100 

In [13]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9457
Recall: 0.9591
F1-Score: 0.9517
              precision    recall  f1-score   support

           0     0.9962    0.9958    0.9960      2397
           1     0.9975    0.9924    0.9949      1980
           2     0.9875    0.9786    0.9830      4023
           3     0.8284    0.9231    0.8732       481
           4     0.9497    0.8750    0.9108       992
           5     0.9669    0.9915    0.9791       118
           6     0.9223    0.9406    0.9314       303
           7     0.9167    0.9757    0.9453       948

    accuracy                         0.9721     11242
   macro avg     0.9457    0.9591    0.9517     11242
weighted avg     0.9730    0.9721    0.9722     11242



- 7100 new corrected

In [6]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9528
Recall: 0.9599
F1-Score: 0.9560
              precision    recall  f1-score   support

           0     0.9975    0.9942    0.9958      2397
           1     0.9960    0.9939    0.9949      1980
           2     0.9847    0.9876    0.9861      3964
           3     0.9048    0.9131    0.9089       541
           4     0.9500    0.8810    0.9142       992
           5     0.9590    0.9915    0.9750       118
           6     0.9114    0.9505    0.9305       303
           7     0.9188    0.9673    0.9424       948

    accuracy                         0.9745     11243
   macro avg     0.9528    0.9599    0.9560     11243
weighted avg     0.9747    0.9745    0.9744     11243



In [15]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9424
Recall: 0.9574
F1-Score: 0.9491
              precision    recall  f1-score   support

           0     0.9971    0.9925    0.9948      2397
           1     0.9954    0.9939    0.9947      1980
           2     0.9872    0.9764    0.9818      4023
           3     0.8107    0.9168    0.8605       481
           4     0.9529    0.8770    0.9134       992
           5     0.9748    0.9831    0.9789       118
           6     0.9022    0.9439    0.9226       303
           7     0.9186    0.9757    0.9463       948

    accuracy                         0.9707     11242
   macro avg     0.9424    0.9574    0.9491     11242
weighted avg     0.9720    0.9707    0.9710     11242



- 8751 

In [6]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9560
Recall: 0.9556
F1-Score: 0.9554
              precision    recall  f1-score   support

           0     0.9966    0.9963    0.9965      2961
           1     0.9936    0.9898    0.9917      2349
           2     0.9808    0.9837    0.9823      4842
           3     0.8692    0.8809    0.8750       747
           4     0.9523    0.8758    0.9125      1232
           5     0.9820    0.9880    0.9850       166
           6     0.9585    0.9493    0.9539       414
           7     0.9147    0.9811    0.9467      1214

    accuracy                         0.9711     13925
   macro avg     0.9560    0.9556    0.9554     13925
weighted avg     0.9714    0.9711    0.9710     13925



In [6]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9551
Recall: 0.9574
F1-Score: 0.9562
              precision    recall  f1-score   support

           0     0.9966    0.9966    0.9966      2961
           1     0.9953    0.9817    0.9884      2349
           2     0.9808    0.9816    0.9812      4842
           3     0.8657    0.8715    0.8686       747
           4     0.9333    0.9083    0.9206      1232
           5     0.9880    0.9880    0.9880       166
           6     0.9499    0.9614    0.9556       414
           7     0.9312    0.9703    0.9504      1214

    accuracy                         0.9709     13925
   macro avg     0.9551    0.9574    0.9562     13925
weighted avg     0.9711    0.9709    0.9709     13925



# Roberta results

In [12]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9533
Recall: 0.9542
F1-Score: 0.9533
              precision    recall  f1-score   support

           0     0.9973    0.9956    0.9965      2961
           1     0.9938    0.9877    0.9907      1460
           2     0.9732    0.9814    0.9773      3924
           3     0.8744    0.8679    0.8711       666
           4     0.9558    0.8644    0.9078       951
           5     0.9771    1.0000    0.9884       171
           6     0.9384    0.9612    0.9496       412
           7     0.9167    0.9755    0.9452       981

    accuracy                         0.9687     11526
   macro avg     0.9533    0.9542    0.9533     11526
weighted avg     0.9689    0.9687    0.9685     11526



- 9500 results

In [6]:
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9577
Recall: 0.9573
F1-Score: 0.9574
              precision    recall  f1-score   support

           0     0.9975    0.9951    0.9963      3266
           1     0.9889    0.9931    0.9910      2329
           2     0.9799    0.9805    0.9802      5382
           3     0.8941    0.9023    0.8982       870
           4     0.9239    0.8804    0.9016      1296
           5     0.9945    0.9891    0.9918       183
           6     0.9454    0.9537    0.9496       454
           7     0.9376    0.9642    0.9507      1590

    accuracy                         0.9703     15370
   macro avg     0.9577    0.9573    0.9574     15370
weighted avg     0.9702    0.9703    0.9702     15370



In [12]:
#roberta 
from sklearn.metrics import classification_report

# Convert to flat lists for metrics calculation
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

precision = precision_score(flat_true_labels, flat_predictions, average='macro')
recall = recall_score(flat_true_labels, flat_predictions, average='macro')
f1 = f1_score(flat_true_labels, flat_predictions, average='macro')

print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

report = classification_report(flat_true_labels, flat_predictions, digits=4)
print(report)

Precision: 0.9585
Recall: 0.9563
F1-Score: 0.9574
              precision    recall  f1-score   support

           0     0.9960    0.9960    0.9960      3266
           1     0.9882    0.9895    0.9888      1523
           2     0.9790    0.9796    0.9793      4370
           3     0.8879    0.8938    0.8908       753
           4     0.9287    0.9048    0.9166      1008
           5     0.9891    0.9785    0.9838       186
           6     0.9487    0.9424    0.9455       451
           7     0.9508    0.9662    0.9584      1301

    accuracy                         0.9714     12858
   macro avg     0.9585    0.9563    0.9574     12858
weighted avg     0.9714    0.9714    0.9713     12858

